In [ ]:
import pandas as pd
import os
import librosa
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix
from sklearn.metrics import r2_score, roc_auc_score, roc_curve, classification_report
from sklearn.svm import SVC
from

#### 893

In [ ]:
train = pd.read_csv('......./metaData_CSS.csv')
train

In [ ]:
train['newlabel'] = ''
i = 0
while i < len(train):
    if train['label'][i] == 'positive':
        train['newlabel'][i] = 1
    elif train['label'][i] == 'negative':
        train['newlabel'][i] = 0
    i = i+1

In [ ]:
train['newlabel'].value_counts()

In [ ]:
df_name  = train['filename']
df_label = np.int64(train['newlabel'])

In [ ]:
os.chdir(r"......./0810/0429/2")
data = '3second_noslience'
filelist = [f for f in os.listdir(data) if os.path.isfile(os.path.join(data, f))]

 
extracted_features=[]
num=1

for file in filelist:
    audio_dataset_path = '......./0810/0429/2/3second_noslience/'
    audio, sample_rate = librosa.load(audio_dataset_path+str(file), res_type='kaiser_fast') 
    lenth = librosa.get_duration(filename=audio_dataset_path+str(file))
    label = file[:1]
    extracted_features.append([lenth,file,label])
    
    print('\r',num,'      ',end='',flush = True)
    num+=1

In [ ]:
info_893_3second_noslience=pd.DataFrame(extracted_features,columns=['lenth','name','label'])
info_893_3second_noslience

In [ ]:
info_1000 = info_893_3second_noslience[info_893_3second_noslience['lenth']>1]
info_1000

##### Coeeficients

In [ ]:
import numpy as np
from tqdm import tqdm
from spafe.features.lfcc import lfcc
from scipy.io.wavfile import read
from spafe.features.lfcc import lfcc
from spafe.utils.preprocessing import SlidingWindow
from spafe.utils.vis import show_features

from spafe.features.mfcc import mfcc

def mfcc_features_extractor(file):
    fs, sig = read(file)
    mfccs  = mfcc(sig,
              fs=fs,
              pre_emph=1,
              pre_emph_coeff=0.97,
              window=SlidingWindow(0.025, 0.01, "hamming"),
              nfilts=80,
              nfft=512,
              low_freq=0,
              high_freq=fs/2,
              #normalize="mvn",
              num_ceps=80,
             )
    mfccs_scaled_features = np.mean(mfccs,axis=0)
    
    return mfccs_scaled_features

In [ ]:
import numpy as np
from tqdm import tqdm
import os
import librosa

audio_dataset_path = '......./0810/0429/2/3second_noslience'



extracted_features=[]
for index_num,row in tqdm(info_1000.iterrows()):
    file_name = os.path.join(os.path.abspath(audio_dataset_path),str(row["name"]))
    final_class_labels=row["label"]
    data=mfcc_features_extractor(file_name)
    name = row["name"]
    audio, sample_rate = librosa.load(file_name , res_type='kaiser_fast') 
    length =  librosa.get_duration(y=audio, sr=sample_rate)
    extracted_features.append([data,final_class_labels,name,length])

In [ ]:
df_features = pd.DataFrame(extracted_features,columns=['feature','label','name','length'])
df_features

In [ ]:
df_features.to_pickle('......./0810/1031/893/coeeficients/25frame_10hop_filter80_mel80.pkl')

In [ ]:
df_frame_length_25 = pd.read_pickle('......./0810/1031/893/coeeficients/25frame_10hop_filter80_mel80.pkl')
i = 0
df_frame_length_25['source'] = ''
while i <len(df_frame_length_25):
    df_frame_length_25['source'][i] = df_frame_length_25['name'][i].split('-')[1]
    i = i+1

In [ ]:
df_frame_length_25['new_column'] = df_frame_length_25['feature'].apply(lambda x: x[:80])

In [ ]:
from sklearn import metrics

kf = StratifiedKFold(n_splits=10, random_state=0, shuffle=True )
list40 = []
for i in range(0,80):
        list40.append(str(i)) 
scaler = StandardScaler()

acc_arr = np.empty((10, 1))
f1_arr = np.empty((10, 1))
precision_arr = np.empty((10, 1))
recall_arr = np.empty((10, 1))
auc_arr = np.empty((10, 1))

sen_arr = np.empty((10, 1))
spe_arr = np.empty((10, 1))

ppv_arr = np.empty((10, 1))
npv_arr = np.empty((10, 1))

recall_arr = np.empty((10, 1))
pre_arr = np.empty((10, 1))

cnf_arr= []
x = 0

for train_index, test_index in kf.split(df_name, df_label):
    name_train, name_test =  df_name[train_index] ,  df_name[test_index] 
    y_train, y_test =  df_label[train_index] ,  df_label[test_index] 
    listname_train =  name_train.values.tolist()
    listname_test =  name_test.values.tolist()
    cut_listname_train = '|'.join(listname_train)
    cut_listname_test  = '|'.join(listname_test)
    train = df_frame_length_25[df_frame_length_25["name"].str.contains(cut_listname_train)]
    test  = df_frame_length_25[df_frame_length_25["name"].str.contains(cut_listname_test)]
    
    features_train = pd.DataFrame(train['new_column'].to_list(), columns= list40)
    features_test  = pd.DataFrame(test['new_column'].to_list(), columns= list40)
    df_x_train = features_train[list40]
    df_x_test  = features_test[list40]
    df_x_train = scaler.fit_transform(df_x_train)
    df_x_test  = scaler.fit_transform(df_x_test)
    
    df_y_train = np.int64(train['label'])
    df_y_test  = np.int64(test['label'])
    model = SVC(kernel='rbf',gamma=0.1, C= 1, tol=1e-5, verbose=1,max_iter=2500) 
    model.fit(df_x_train, df_y_train)
    y_pred = model.predict(df_x_test)
    print('Accuracy Score: {:.4f}'.format(accuracy_score(df_y_test, y_pred)))
    print('SVC f1-score  : {:.4f}'.format(f1_score(df_y_test, y_pred)))
    print('SVC precision : {:.4f}'.format(precision_score(df_y_test, y_pred)))
    print('SVC recall    : {:.4f}'.format(recall_score(df_y_test, y_pred)))
    print("\n", classification_report(df_y_test, y_pred))
    
    cnf_matrix = confusion_matrix(df_y_test, y_pred)
    acc_arr[x] = accuracy_score(df_y_test, y_pred)
    f1_arr[x] = f1_score(df_y_test, y_pred)
    precision_arr[x] = precision_score(df_y_test, y_pred)
    recall_arr[x] = recall_score(df_y_test, y_pred)
    auc_arr[x] = metrics.roc_auc_score(df_y_test, y_pred)
    
    tn, fp, fn, tp = confusion_matrix(df_y_test, y_pred).ravel()
    specificity = tn / (tn+fp)
    sensitivity = tp / (tp+fn)
    ppv = tp /(tp+fp)
    npv = tn /(tn+fn)
    pre = tp / (tp + fp)
    
    sen_arr[x] = sensitivity
    spe_arr[x] = specificity
    ppv_arr[x] = ppv
    npv_arr[x] = npv
    pre_arr[x] = pre
    
     

    x = x+ 1


print("%0.3f accuracy with a standard deviation of %0.2f" %
      (acc_arr.mean(), acc_arr.std()))
print("%0.3f precision with a standard deviation of %0.2f" %
      (precision_arr.mean(), precision_arr.std()))
print("%0.3f auc with a standard deviation of %0.2f" %
      (auc_arr.mean(), auc_arr.std()))
print("%0.3f f1 score with a standard deviation of %0.2f" %
      (f1_arr.mean(), f1_arr.std()))

print("%0.3f sensitivity with a standard deviation of %0.2f" %
      (sen_arr.mean(), sen_arr.std()))

print("%0.3f specificity with a standard deviation of %0.2f" %
      (spe_arr.mean(), spe_arr.std()))

print("%0.3f ppv with a standard deviation of %0.2f" %
      (ppv_arr.mean(), ppv_arr.std()))
print("%0.3f npv with a standard deviation of %0.2f" %
      (npv_arr.mean(), npv_arr.std()))

print("%0.3f recall with a standard deviation of %0.2f" %
      (recall_arr.mean(), recall_arr.std()))

In [ ]:
acc_arr